# Eksperimen: ARIMA N_WINDOW=24 vs ARIMA Tanpa Window (Full Data)

Notebook ini menjawab komentar dosen:
> 'Kenapa ARIMA hanya pakai 24 bulan padahal LSTM menggunakan semua data?
  Apakah tidak bisa ARIMA tanpa batasan 24 bulan? Apa bedanya?'

**Konfigurasi A**: ARIMA dengan N_WINDOW=24 (pipeline original Laura)
**Konfigurasi B**: ARIMA tanpa batasan window (full data per produk)

PENTING: Seluruh logika preprocessing, IQR clipping, filter produk, bias correction,
momentum, cap, dan evaluasi rolling IDENTIK dengan pipeline original (Notebook v5 Laura).
Satu-satunya perbedaan adalah ada/tidaknya `.tail(n_window)` saat menyiapkan data ARIMA.

In [1]:
## Sel 0 -- Setup: identik dengan Sel 0 notebook original Laura
## Perubahan: tambah N_WINDOW_A dan N_WINDOW_B untuk eksperimen
import os, random, time, warnings, tracemalloc
from collections import Counter
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ.setdefault('PYTHONHASHSEED', '0')

import pandas as pd
import numpy as np
import tensorflow as tf
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from joblib import Parallel, delayed

# ── Semua konstanta identik dengan pipeline original ────────────────────────
csv_path         = 'dataset_toko.csv'
SPLIT_PCT        = 0.80
MIN_BULAN        = 15
RANDOM_SEED      = 42
SEQ_LEN          = 6
N_CLUSTER        = 5
BIAS_HOLDOUT     = 7
CAP_FACTOR_ARIMA = 1.0   # <-- identik dengan Sel 0 notebook original
CAP_FACTOR_LSTM  = 1.5
MIN_BULAN_AKTIF  = 3
IQR_MULTIPLIER   = 1.5
TOP_N            = 10
N_JOBS           = -1
ARIMA_MAXITER    = 100
ARIMA_ORDERS = [
    (1,1,1),(1,1,0),(0,1,1),(2,1,0),
    (0,1,2),(2,1,1),(1,1,2),(3,1,0),
    (0,1,3),(2,1,2),(1,2,1),(0,2,1),
]
KALENDER_LIBUR = {
    '2020-05':0.40,'2021-05':0.45,'2022-05':0.55,'2023-04':0.50,'2024-04':0.65,'2025-03':0.90,
    '2020-06':0.60,'2021-06':0.60,'2022-06':0.65,'2020-07':0.60,'2021-07':0.65,'2022-07':0.70,
    '2023-06':0.65,'2024-05':0.60,'2024-06':0.12,'2024-07':0.28,'2024-08':0.55,
    '2024-12':0.40,'2025-01':0.58,
}
BULAN_EKSKLUDE = ['2024-06','2024-07']
BULAN_ANOMALI  = BULAN_EKSKLUDE

# ── TAMBAHAN untuk eksperimen (tidak ada di pipeline original) ───────────────
N_WINDOW_A = 24    # Konfigurasi A: identik dengan N_WINDOW_ARIMA pipeline original
N_WINDOW_B = None  # Konfigurasi B: tanpa batas window, gunakan semua data training

def set_seed_ulang(offset=0):
    random.seed(RANDOM_SEED + offset)
    np.random.seed(RANDOM_SEED + offset)
    tf.random.set_seed(RANDOM_SEED + offset)
set_seed_ulang(0)

print('Setup selesai.')
print(f'CAP_FACTOR_ARIMA = {CAP_FACTOR_ARIMA} (identik dengan pipeline original)')
print(f'Konfigurasi A: N_WINDOW = {N_WINDOW_A} bulan terakhir (pipeline original)')
print(f'Konfigurasi B: N_WINDOW = {N_WINDOW_B} = gunakan SELURUH data training per produk')

Setup selesai.
CAP_FACTOR_ARIMA = 1.0 (identik dengan pipeline original)
Konfigurasi A: N_WINDOW = 24 bulan terakhir (pipeline original)
Konfigurasi B: N_WINDOW = None = gunakan SELURUH data training per produk


In [ ]:
## Sel 1 -- Load, Bersihkan, Agregasi, IQR, Filter Produk
## IDENTIK dengan Sel 4, 5, 6 notebook original Laura

# ── Load & Parse ─────────────────────────────────────────────────────────────
df_raw = pd.read_csv(csv_path, on_bad_lines='skip')
df = df_raw.copy()
df['Tanggal Pembayaran'] = pd.to_datetime(
    df['Tanggal Pembayaran'], format='mixed', errors='coerce')
df = df.dropna(subset=['Tanggal Pembayaran'])
df = df[df['Status Terakhir'] == 'Pesanan Selesai'].copy()
for col in ['Harga Jual (IDR)', 'Jumlah Produk Dibeli']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
df['item_revenue'] = (df['Harga Jual (IDR)'] * df['Jumlah Produk Dibeli']).clip(lower=0)
df['bulan_period'] = df['Tanggal Pembayaran'].dt.to_period('M')
df['bulan']        = df['Tanggal Pembayaran'].dt.month

harga_rata2 = df.groupby('Nama Produk')['Harga Jual (IDR)'].mean().to_dict()
bulan_list  = sorted(df['bulan_period'].unique())
split_idx   = int(len(bulan_list) * SPLIT_PCT)
bulan_train = bulan_list[:split_idx]
bulan_test  = bulan_list[split_idx:]

# ── Agregasi ──────────────────────────────────────────────────────────────────
monthly_all = (
    df.groupby(['bulan_period','bulan','Nama Produk'])
      .agg(qty=('Jumlah Produk Dibeli','sum'), revenue=('item_revenue','sum'))
      .reset_index().sort_values(['Nama Produk','bulan_period'])
)

# ── IQR Clipping (leak-free, identik dengan Sel 5 original) ───────────────────
monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)].copy()
monthly_test  = monthly_all[monthly_all['bulan_period'].isin(bulan_test)].copy()
iqr_bounds = {}
for p in monthly_train['Nama Produk'].unique():
    vals = monthly_train[monthly_train['Nama Produk']==p]['qty']
    if len(vals) < 4: continue
    Q1, Q3 = vals.quantile(0.25), vals.quantile(0.75)
    IQR = Q3 - Q1
    iqr_bounds[p] = (max(0.0, Q1-IQR_MULTIPLIER*IQR), Q3+IQR_MULTIPLIER*IQR)
def _clip_qty(row):
    b = iqr_bounds.get(row['Nama Produk'])
    return row['qty'] if b is None else float(np.clip(row['qty'], b[0], b[1]))
monthly_all['qty'] = monthly_all.apply(_clip_qty, axis=1)
monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)].copy()
monthly_test  = monthly_all[monthly_all['bulan_period'].isin(bulan_test)].copy()

# ── Filter Produk Layak (identik dengan Sel 6 original) ───────────────────────
produk_count      = monthly_train.groupby('Nama Produk')['bulan_period'].count()
produk_layak_awal = produk_count[produk_count >= MIN_BULAN].index.tolist()
bulan_train_bersih = [b for b in bulan_train if str(b) not in BULAN_ANOMALI]
bulan_cek_aktif    = bulan_train_bersih[-MIN_BULAN_AKTIF:]
def cek_aktif(p):
    df_p = monthly_train[monthly_train['Nama Produk']==p]
    return df_p[df_p['bulan_period'].isin(bulan_cek_aktif)]['qty'].sum() > 0
produk_layak = [p for p in produk_layak_awal if cek_aktif(p)]

mean_qty_produk, median_qty_produk, max_qty_produk = {}, {}, {}
for p in produk_layak:
    vals = monthly_train[
        (monthly_train['Nama Produk']==p) &
        (~monthly_train['bulan_period'].astype(str).isin(BULAN_ANOMALI))
    ]['qty'].values
    mean_qty_produk[p]   = max(float(vals.mean()), 1.0) if len(vals)>0 else 1.0
    median_qty_produk[p] = max(float(np.median(vals)), 1.0) if len(vals)>0 else 1.0
    max_qty_produk[p]    = max(float(vals.max()), 1.0) if len(vals)>0 else 1.0

print(f'Data siap. {len(produk_layak)} produk layak dimodelkan.')
print(f'Training: {bulan_train[0]} s.d. {bulan_train[-1]} ({len(bulan_train)} bulan)')
print(f'Testing : {bulan_test[0]} s.d. {bulan_test[-1]} ({len(bulan_test)} bulan)')

Data siap. 166 produk layak dimodelkan.
Training: 2020-05 s.d. 2024-03 (47 bulan)
Testing : 2024-04 s.d. 2025-03 (12 bulan)


In [3]:
## Sel 2 -- Fungsi Helper ARIMA (versi generalisasi dari Sel 1-3 original)
## PERBEDAAN SATU-SATUNYA: parameter n_window pada dua fungsi di bawah.
## Jika n_window=None -> gunakan semua data (tidak ada .tail()).
## Semua logika lain (log transform, grid search AIC, bias correction,
## momentum, cap, fallback WMA) identik dengan fungsi asli Laura.

def hitung_momentum(df_c):
    """Identik dengan fungsi asli."""
    if len(df_c) < 3:
        return 1.0
    vals = df_c['qty'].values[-3:]
    if vals[0] > 0:
        tren = (vals[-1] - vals[0]) / vals[0]
        return float(np.clip(1.0 + tren * 0.10, 0.90, 1.10))
    return 1.0


def _latih_satu_produk_arima_v(produk, monthly_train, n_window):
    """
    Identik dengan _latih_satu_produk_arima() asli, KECUALI:
    - Jika n_window tidak None: df_c = df_c.tail(n_window)  [Konfigurasi A]
    - Jika n_window adalah None: df_c tidak dipotong          [Konfigurasi B]
    """
    df_c = monthly_train[
        monthly_train['Nama Produk'] == produk
    ].sort_values('bulan_period')
    df_c = df_c[~df_c['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)]

    # ── PERBEDAAN UTAMA: satu baris ini ──────────────────────────────────────
    if n_window is not None:
        df_c = df_c.tail(n_window)   # Konfigurasi A: potong ke n_window bulan terakhir
    # else: Konfigurasi B — tidak dipotong, gunakan semua bulan yang ada
    # ─────────────────────────────────────────────────────────────────────────

    if len(df_c) < 10:
        return produk, None, 1.0

    # Log transform (identik dengan asli)
    fl_v = df_c['bulan_period'].astype(str).map(
        lambda x: KALENDER_LIBUR.get(x, 1.0)).values
    ts_log = np.log1p(
        df_c['qty'].clip(lower=0.1).values / np.maximum(fl_v, 0.1))

    # Grid search AIC (identik dengan asli)
    best_aic, best_order = np.inf, (1, 1, 1)
    for order in ARIMA_ORDERS:
        try:
            m = ARIMA(ts_log, order=order,
                      enforce_stationarity=True,
                      enforce_invertibility=True).fit(
                method_kwargs={'maxiter': ARIMA_MAXITER})
            if m.aic < best_aic:
                best_aic, best_order = m.aic, order
        except Exception:
            pass

    # Bias correction (identik dengan asli)
    bias = 1.0
    if len(df_c) > BIAS_HOLDOUT + 8:
        try:
            tr_b = df_c.iloc[:-BIAS_HOLDOUT]
            ho_b = df_c.iloc[-BIAS_HOLDOUT:]
            ts_tb = np.log1p(
                tr_b['qty'].clip(lower=0.1).values /
                np.maximum(
                    tr_b['bulan_period'].astype(str).map(
                        lambda x: KALENDER_LIBUR.get(x, 1.0)).values,
                    0.1))
            m_b = ARIMA(ts_tb, order=best_order,
                        enforce_stationarity=True,
                        enforce_invertibility=True).fit(
                method_kwargs={'maxiter': ARIMA_MAXITER})
            fc_log = np.clip(m_b.forecast(steps=BIAS_HOLDOUT), -2.0, 10.0)
            fl_hb  = ho_b['bulan_period'].astype(str).map(
                lambda x: KALENDER_LIBUR.get(x, 1.0)).values
            fc_qty = np.expm1(fc_log) * fl_hb
            rasio  = max(float(np.asarray(fc_qty).sum()), 1e-6) / \
                     float(ho_b['qty'].values.sum())
            bias   = float(np.clip(rasio, 0.7, 1.5))
        except Exception:
            bias = 1.0

    return produk, best_order, bias


def latih_semua_arima_v(produk_list, monthly_train, n_window):
    """Identik dengan latih_semua_arima() asli, meneruskan n_window."""
    hasil_paralel = Parallel(n_jobs=N_JOBS, backend='loky')(
        delayed(_latih_satu_produk_arima_v)(p, monthly_train, n_window)
        for p in produk_list)
    best_orders, bias_corr = {}, {}
    for p, order, bias in hasil_paralel:
        bias_corr[p] = bias
        if order is not None:
            best_orders[p] = order
    return best_orders, bias_corr


def _prediksi_satu_produk_arima_v(produk, bulan_str, faktor_libur,
                                   monthly_avail, best_orders_arima,
                                   bias_correction, harga_rata2,
                                   max_qty_produk, n_window):
    """
    Identik dengan _prediksi_satu_produk_arima() asli, KECUALI:
    - Potongan tail(n_window) hanya dilakukan jika n_window tidak None.
    """
    df_p = monthly_avail[
        monthly_avail['Nama Produk'] == produk
    ].sort_values('bulan_period')
    df_c = df_p[~df_p['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)]

    # ── PERBEDAAN UTAMA ───────────────────────────────────────────────────────
    if n_window is not None and len(df_c) > n_window:
        df_c = df_c.tail(n_window)
    # ─────────────────────────────────────────────────────────────────────────

    fc = None
    if produk in best_orders_arima and len(df_c) >= 6:
        try:
            qty_vals = df_c['qty'].clip(lower=0.1).values
            fl_vals  = df_c['bulan_period'].astype(str).map(
                lambda x: KALENDER_LIBUR.get(x, 1.0)).values
            ts_log   = np.log1p(qty_vals / np.maximum(fl_vals, 0.1))
            m = ARIMA(ts_log, order=best_orders_arima[produk],
                      enforce_stationarity=True,
                      enforce_invertibility=True).fit(
                method_kwargs={'maxiter': ARIMA_MAXITER})
            fc_result = m.forecast(steps=1)
            fc_val    = fc_result.iloc[0] if hasattr(fc_result, 'iloc') \
                        else np.asarray(fc_result).reshape(-1)[0]
            fc = max(0.0, float(np.expm1(
                float(np.clip(fc_val, -2.0, 10.0)))) * faktor_libur)
            bias = bias_correction.get(produk, 1.0)
            if bias > 0:
                fc = fc / bias
            fc = fc * hitung_momentum(df_c)
            fc = min(max(0.0, fc),
                     max_qty_produk.get(produk, fc) * CAP_FACTOR_ARIMA)
        except Exception:
            fc = None

    # Fallback WMA (identik dengan asli)
    if fc is None and len(df_c) >= 1:
        recent = df_c['qty'].values[-min(6, len(df_c)):]
        bobot  = np.array([1,2,3,4,5,6][-len(recent):], dtype=float)
        fc     = float(np.average(recent, weights=bobot)) * faktor_libur
    if fc is None:
        fc = float(df_c['qty'].median()) * faktor_libur if len(df_c) > 0 else 0.0

    pred_qty = max(0.0, fc)
    return {
        'nama_produk'    : produk,
        'pred_qty_arima' : round(pred_qty, 2),
        'pred_rev_arima' : round(pred_qty * harga_rata2.get(produk, 0), 0)
    }


def prediksi_arima_v(bulan_pred, monthly_avail, produk_layak,
                     best_orders_arima, bias_correction,
                     harga_rata2, max_qty_produk, n_window):
    """Identik dengan prediksi_arima() asli, meneruskan n_window."""
    bulan_str    = str(bulan_pred)
    faktor_libur = KALENDER_LIBUR.get(bulan_str, 1.0)
    hasil = Parallel(n_jobs=N_JOBS, backend='loky')(
        delayed(_prediksi_satu_produk_arima_v)(
            p, bulan_str, faktor_libur, monthly_avail,
            best_orders_arima, bias_correction,
            harga_rata2, max_qty_produk, n_window)
        for p in produk_layak)
    return pd.DataFrame(hasil) if hasil else pd.DataFrame(
        columns=['nama_produk','pred_qty_arima','pred_rev_arima'])


print('Fungsi helper ARIMA (versi generalisasi) siap.')

Fungsi helper ARIMA (versi generalisasi) siap.


In [4]:
## Sel 3 -- Training Konfigurasi A: N_WINDOW=24 (identik dengan pipeline original)

print('='*65)
print(f'KONFIGURASI A: N_WINDOW_ARIMA = {N_WINDOW_A} bulan terakhir (ORIGINAL)')
print('='*65)
print()
print(f'Setiap produk menggunakan MAKSIMUM {N_WINDOW_A} bulan terakhir')
print(f'dari {len(bulan_train)} bulan data training yang tersedia.')
print(f'Produk dengan data < {N_WINDOW_A} bulan tetap memakai semua datanya.')
print()

t0 = time.time()
best_orders_A, bias_corr_A = latih_semua_arima_v(
    produk_layak, monthly_train, n_window=N_WINDOW_A)
t_A = time.time() - t0

n_berhasil_A  = len(best_orders_A)
order_count_A = Counter(best_orders_A.values())
order_top_A, freq_top_A = order_count_A.most_common(1)[0]

print(f'Hasil Training Konfigurasi A:')
print(f'  Produk berhasil dilatih : {n_berhasil_A} dari {len(produk_layak)}')
print(f'  Waktu training          : {t_A:.1f} detik')
print(f'  Order terpopuler        : {order_top_A} '
      f'({freq_top_A}/{n_berhasil_A} = {freq_top_A/n_berhasil_A*100:.1f}%)')
print()
print('  Distribusi order ARIMA terpopuler (Konfigurasi A):')
display(pd.DataFrame(order_count_A.most_common(5),
                     columns=['Order (p,d,q)', 'Jumlah Produk']))

KONFIGURASI A: N_WINDOW_ARIMA = 24 bulan terakhir (ORIGINAL)

Setiap produk menggunakan MAKSIMUM 24 bulan terakhir
dari 47 bulan data training yang tersedia.
Produk dengan data < 24 bulan tetap memakai semua datanya.

Hasil Training Konfigurasi A:
  Produk berhasil dilatih : 166 dari 166
  Waktu training          : 21.3 detik
  Order terpopuler        : (0, 1, 1) (107/166 = 64.5%)

  Distribusi order ARIMA terpopuler (Konfigurasi A):


,"Order (p,d,q)",Jumlah Produk
0,"(0, 1, 1)",107
1,"(0, 1, 2)",12
2,"(0, 1, 3)",12
3,"(2, 1, 0)",7
4,"(2, 1, 2)",6


In [5]:
## Sel 4 -- Training Konfigurasi B: N_WINDOW=None (full data, tanpa pembatasan)

print('='*65)
print('KONFIGURASI B: N_WINDOW_ARIMA = None (SEMUA DATA per produk)')
print('='*65)
print()
print(f'Setiap produk menggunakan SEMUA data training yang tersedia.')
print(f'Total rentang: {len(bulan_train)} bulan ({bulan_train[0]} s.d. {bulan_train[-1]}).')
print()

# Analisis panjang data aktual per produk
panjang_data = []
for p in produk_layak:
    df_p = monthly_train[monthly_train['Nama Produk']==p]
    df_c = df_p[~df_p['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)]
    panjang_data.append(len(df_c))
panjang_arr = np.array(panjang_data)
n_berbeda   = int((panjang_arr > N_WINDOW_A).sum())
n_identik   = int((panjang_arr <= N_WINDOW_A).sum())
print(f'Dari {len(produk_layak)} produk layak:')
print(f'  {n_berbeda} produk punya data > {N_WINDOW_A} bulan '
      f'-> Konfigurasi B akan BERBEDA dari A untuk produk-produk ini')
print(f'  {n_identik} produk punya data <= {N_WINDOW_A} bulan '
      f'-> Identik dengan Konfigurasi A untuk produk-produk ini')
print(f'  Rentang panjang data: {panjang_arr.min()}–{panjang_arr.max()} bulan '
      f'(median: {np.median(panjang_arr):.0f})')
print()

t0 = time.time()
best_orders_B, bias_corr_B = latih_semua_arima_v(
    produk_layak, monthly_train, n_window=N_WINDOW_B)
t_B = time.time() - t0

n_berhasil_B  = len(best_orders_B)
order_count_B = Counter(best_orders_B.values())
order_top_B, freq_top_B = order_count_B.most_common(1)[0]

print(f'Hasil Training Konfigurasi B:')
print(f'  Produk berhasil dilatih : {n_berhasil_B} dari {len(produk_layak)}')
print(f'  Waktu training          : {t_B:.1f} detik')
print(f'  Order terpopuler        : {order_top_B} '
      f'({freq_top_B}/{n_berhasil_B} = {freq_top_B/n_berhasil_B*100:.1f}%)')
print()
print('  Distribusi order ARIMA terpopuler (Konfigurasi B):')
display(pd.DataFrame(order_count_B.most_common(5),
                     columns=['Order (p,d,q)', 'Jumlah Produk']))
print()

# Produk mana yang memilih order BERBEDA antara A dan B?
perbedaan_order = []
for i, p in enumerate(produk_layak):
    ord_a = best_orders_A.get(p)
    ord_b = best_orders_B.get(p)
    if ord_a != ord_b:
        perbedaan_order.append({
            'Produk'           : p[:50] + ('...' if len(p)>50 else ''),
            'Panjang Data (bln)': panjang_data[i],
            'Order A (24 bln)' : str(ord_a),
            'Order B (full)'   : str(ord_b),
        })
print(f'Produk yang memilih ORDER ARIMA BERBEDA antara A dan B: '
      f'{len(perbedaan_order)} dari {len(produk_layak)}')
if perbedaan_order:
    display(pd.DataFrame(perbedaan_order))
else:
    print('Semua produk memilih order yang sama di kedua konfigurasi.')

KONFIGURASI B: N_WINDOW_ARIMA = None (SEMUA DATA per produk)

Setiap produk menggunakan SEMUA data training yang tersedia.
Total rentang: 47 bulan (2020-05 s.d. 2024-03).

Dari 166 produk layak:
  129 produk punya data > 24 bulan -> Konfigurasi B akan BERBEDA dari A untuk produk-produk ini
  37 produk punya data <= 24 bulan -> Identik dengan Konfigurasi A untuk produk-produk ini
  Rentang panjang data: 15–47 bulan (median: 30)

Hasil Training Konfigurasi B:
  Produk berhasil dilatih : 166 dari 166
  Waktu training          : 21.5 detik
  Order terpopuler        : (0, 1, 1) (111/166 = 66.9%)

  Distribusi order ARIMA terpopuler (Konfigurasi B):


,"Order (p,d,q)",Jumlah Produk
0,"(0, 1, 1)",111
1,"(0, 1, 2)",13
2,"(0, 1, 3)",8
3,"(1, 1, 1)",8
4,"(2, 1, 2)",7



Produk yang memilih ORDER ARIMA BERBEDA antara A dan B: 51 dari 166


,Produk,Panjang Data (bln),Order A (24 bln),Order B (full)
0,AS 6mm KIPAS ANGIN KECIL 7 IN 9 IN BATANG BESI,30,"(1, 1, 1)","(0, 1, 1)"
1,AS EXHAUST PANASONIC NATIONAL DINAMO MASPION E...,38,"(1, 1, 2)","(0, 1, 2)"
2,AS Exhaust/Exaust/Exsos Maspion Panasonic Nati...,29,"(3, 1, 0)","(0, 1, 3)"
3,AS KIPAS ANGIN REGENCY 10 CM DIAMETER 10 MM,30,"(2, 1, 1)","(0, 1, 1)"
4,"AS kipas angin model Maspion 18,5 cm.AS Maspio...",47,"(0, 1, 2)","(0, 1, 1)"
5,"AS kipas angin model Maspion 20,5 cm.AS Maspio...",47,"(0, 1, 3)","(0, 1, 1)"
6,BEARING 629 NKN 2 RS BERING BUSHING BALL LAHER...,32,"(2, 1, 2)","(0, 1, 1)"
7,ELEMEN TUTUP ATAS MAGIC COM JAR PEMANAS TOP HE...,43,"(0, 1, 1)","(2, 1, 2)"
8,GEAR GIGI MIXER MIYAKO HM 620 625 650 GRIGI SE...,43,"(0, 1, 1)","(1, 1, 1)"
9,Gear Kopel Maspion Upper Blender Mounting Baut...,29,"(1, 1, 0)","(0, 1, 1)"


In [6]:
## Sel 5 -- Evaluasi Rolling 12 Bulan: A vs B
## Identik dengan Sel 10 notebook original, dijalankan dua kali (A dan B).

print('='*65)
print('EVALUASI ROLLING 12 BULAN: KONFIGURASI A vs KONFIGURASI B')
print('='*65)
print('(Walk-forward validation identik dengan pipeline original)')
print()

def jalankan_rolling_arima(label, best_orders, bias_corr, n_window):
    """Evaluasi rolling identik dengan Sel 10 original."""
    log_bulan, semua_hasil_lokal = [], []
    for bulan_pred in bulan_test:
        avail = monthly_all[monthly_all['bulan_period'] < bulan_pred]
        df_pred = prediksi_arima_v(
            bulan_pred, avail, produk_layak,
            best_orders, bias_corr,
            harga_rata2, max_qty_produk, n_window)
        aktual_b = monthly_test[
            monthly_test['bulan_period'] == bulan_pred
        ][['Nama Produk','qty','revenue']].rename(
            columns={'Nama Produk':'nama_produk',
                     'qty':'aktual_qty',
                     'revenue':'aktual_rev'})
        df_m = pd.merge(df_pred, aktual_b, on='nama_produk', how='left').fillna(0)
        semua_hasil_lokal.append(df_m)
        log_bulan.append({
            'bulan'   : str(bulan_pred),
            'aktual'  : df_m['aktual_rev'].sum(),
            'pred'    : df_m['pred_rev_arima'].sum(),
        })
    return pd.DataFrame(log_bulan), semua_hasil_lokal

# Jalankan Konfigurasi A
t0 = time.time()
df_log_A, hasil_A = jalankan_rolling_arima(
    'A', best_orders_A, bias_corr_A, N_WINDOW_A)
print(f'Konfigurasi A (N_WINDOW={N_WINDOW_A}) selesai: {time.time()-t0:.1f} detik')

# Jalankan Konfigurasi B
t0 = time.time()
df_log_B, hasil_B = jalankan_rolling_arima(
    'B', best_orders_B, bias_corr_B, N_WINDOW_B)
print(f'Konfigurasi B (N_WINDOW=None) selesai : {time.time()-t0:.1f} detik')
print()

# Tampilkan prediksi per bulan berdampingan
df_banding = df_log_A[['bulan','aktual','pred']].copy()
df_banding = df_banding.rename(columns={'pred': 'pred_A'})
df_banding['pred_B'] = df_log_B['pred'].values
df_banding['error_A'] = (df_banding['pred_A'] - df_banding['aktual']).abs()
df_banding['error_B'] = (df_banding['pred_B'] - df_banding['aktual']).abs()
df_banding['unggul'] = df_banding.apply(
    lambda r: 'A (window-24)' if r.error_A < r.error_B
    else ('B (full-data)' if r.error_B < r.error_A else 'Seri'), axis=1)

print('Revenue aktual vs prediksi per bulan (satuan Rupiah):')
display(df_banding)

EVALUASI ROLLING 12 BULAN: KONFIGURASI A vs KONFIGURASI B
(Walk-forward validation identik dengan pipeline original)

Konfigurasi A (N_WINDOW=24) selesai: 20.1 detik
Konfigurasi B (N_WINDOW=None) selesai : 17.5 detik

Revenue aktual vs prediksi per bulan (satuan Rupiah):


,bulan,aktual,pred_A,pred_B,error_A,error_B,unggul
0,2024-04,12248900.0,13201450.0,14387312.0,952550.0,2138412.0,A (window-24)
1,2024-05,6114200.0,11587701.0,11916905.0,5473501.0,5802705.0,A (window-24)
2,2024-06,1745800.0,2105448.0,2078101.0,359648.0,332301.0,B (full-data)
3,2024-07,3420200.0,4912725.0,4848919.0,1492525.0,1428719.0,B (full-data)
4,2024-08,5895225.0,9649998.0,9524664.0,3754773.0,3629439.0,B (full-data)
5,2024-09,12046400.0,17424782.0,16963993.0,5378382.0,4917593.0,B (full-data)
6,2024-10,14812400.0,16959614.0,17139956.0,2147214.0,2327556.0,A (window-24)
7,2024-11,14015450.0,16306065.0,16123776.0,2290615.0,2108326.0,B (full-data)
8,2024-12,4747175.0,6094097.0,6361391.0,1346922.0,1614216.0,A (window-24)
9,2025-01,7832075.0,8933440.0,9001013.0,1101365.0,1168938.0,A (window-24)


In [7]:
## Sel 6 -- Metrik MAE, RMSE, dan Precision@10: A vs B

aktual_arr = df_log_A['aktual'].values
pred_A_arr = df_log_A['pred'].values
pred_B_arr = df_log_B['pred'].values

mae_A  = float(mean_absolute_error(aktual_arr, pred_A_arr))
rmse_A = float(np.sqrt(mean_squared_error(aktual_arr, pred_A_arr)))
mae_B  = float(mean_absolute_error(aktual_arr, pred_B_arr))
rmse_B = float(np.sqrt(mean_squared_error(aktual_arr, pred_B_arr)))

# Precision@10 per bulan
prec_A_list, prec_B_list = [], []
for df_a, df_b in zip(hasil_A, hasil_B):
    top_aktual = set(df_a.nlargest(TOP_N, 'aktual_rev')['nama_produk'])
    top_a      = set(df_a.nlargest(TOP_N, 'pred_rev_arima')['nama_produk'])
    top_b      = set(df_b.nlargest(TOP_N, 'pred_rev_arima')['nama_produk'])
    prec_A_list.append(len(top_a & top_aktual) / TOP_N)
    prec_B_list.append(len(top_b & top_aktual) / TOP_N)

prec_A = np.mean(prec_A_list)
prec_B = np.mean(prec_B_list)

# Tabel ringkasan
df_ringkasan = pd.DataFrame({
    'Konfigurasi': [
        f'A — N_WINDOW={N_WINDOW_A} bulan (pipeline original Laura)',
        'B — N_WINDOW=None (full data, apple-to-apple dengan LSTM)'
    ],
    'MAE (Rp)'      : [f'{mae_A:,.0f}', f'{mae_B:,.0f}'],
    'RMSE (Rp)'     : [f'{rmse_A:,.0f}', f'{rmse_B:,.0f}'],
    'Precision@10'  : [f'{prec_A:.3f}', f'{prec_B:.3f}'],
    'Waktu Training': [f'{t_A:.1f} dtk', f'{t_B:.1f} dtk'],
})
print('='*70)
print('RINGKASAN METRIK: Konfigurasi A vs Konfigurasi B')
print('='*70)
display(df_ringkasan)

# Precision@10 per bulan berdampingan
df_prec = pd.DataFrame({
    'bulan'       : [str(b) for b in bulan_test],
    'overlap_A'   : [round(p*TOP_N) for p in prec_A_list],
    'prec10_A'    : prec_A_list,
    'overlap_B'   : [round(p*TOP_N) for p in prec_B_list],
    'prec10_B'    : prec_B_list,
    'unggul'      : ['A' if a>b else ('B' if b>a else 'Seri')
                     for a,b in zip(prec_A_list, prec_B_list)]
})
print()
print('Precision@10 per bulan:')
display(df_prec)
print(f'Rata-rata Precision@10 — A: {prec_A:.3f} | B: {prec_B:.3f}')

RINGKASAN METRIK: Konfigurasi A vs Konfigurasi B


,Konfigurasi,MAE (Rp),RMSE (Rp),Precision@10,Waktu Training
0,A — N_WINDOW=24 bulan (pipeline original Laura),"2,467,639","2,950,435",0.358,21.3 dtk
1,"B — N_WINDOW=None (full data, apple-to-apple d...","2,462,224","2,899,922",0.342,21.5 dtk



Precision@10 per bulan:


,bulan,overlap_A,prec10_A,overlap_B,prec10_B,unggul
0,2024-04,3,0.3,3,0.3,Seri
1,2024-05,2,0.2,1,0.1,A
2,2024-06,2,0.2,1,0.1,A
3,2024-07,3,0.3,2,0.2,A
4,2024-08,4,0.4,3,0.3,A
5,2024-09,4,0.4,5,0.5,B
6,2024-10,5,0.5,5,0.5,Seri
7,2024-11,4,0.4,3,0.3,A
8,2024-12,4,0.4,5,0.5,B
9,2025-01,5,0.5,6,0.6,B


Rata-rata Precision@10 — A: 0.358 | B: 0.342


In [8]:
## Sel 7 -- Kesimpulan Otomatis dan Rekomendasi untuk Skripsi

print('='*70)
print('KESIMPULAN DAN REKOMENDASI')
print('='*70)
print()

# Kemenangan per bulan
unggul_count = df_banding['unggul'].value_counts()
print('Kemenangan per bulan (berdasarkan error absolut terkecil):')
for label, count in unggul_count.items():
    print(f'  {label}: {count} bulan')
print()

# Perbandingan MAE
print('Perbandingan MAE:')
print(f'  Konfigurasi A (N_WINDOW=24)   : Rp{mae_A:,.0f}')
print(f'  Konfigurasi B (N_WINDOW=None) : Rp{mae_B:,.0f}')
print()
if mae_A < mae_B:
    selisih = (mae_B - mae_A) / mae_B * 100
    print(f'  -> Konfigurasi A (window-24) LEBIH BAIK ({selisih:.1f}% MAE lebih rendah).')
    print()
    print('INTERPRETASI:')
    print('  Membatasi window ke 24 bulan TERBUKTI secara empiris menghasilkan')
    print('  MAE lebih rendah. Artinya: data training sebelum 24 bulan terakhir')
    print('  (terutama pola 2020-2021 saat awal pandemi) tidak membantu —')
    print('  bahkan mengganggu — prediksi ARIMA untuk kondisi terkini.')
    print()
    print('REKOMENDASI UNTUK SKRIPSI:')
    print('  Pertahankan N_WINDOW=24. Tambahkan tabel eksperimen ini (Tabel baru')
    print('  di Subbab 4.6.2) sebagai justifikasi empiris atas pemilihan N_WINDOW=24.')
    print('  Penjelasan: meskipun LSTM menggunakan full data dan ARIMA dibatasi 24 bulan,')
    print('  eksperimen ini membuktikan bahwa pembatasan tersebut justru MENINGKATKAN')
    print('  performa ARIMA — sehingga penggunaan 24 bulan bukan kelemahan melainkan')
    print('  merupakan konfigurasi optimal yang ditentukan secara empiris.')
elif mae_B < mae_A:
    selisih = (mae_A - mae_B) / mae_A * 100
    print(f'  -> Konfigurasi B (full data) LEBIH BAIK ({selisih:.1f}% MAE lebih rendah).')
    print()
    print('INTERPRETASI:')
    print('  Menggunakan full data memberikan prediksi lebih akurat untuk ARIMA.')
    print('  Ini sekaligus membuat perbandingan dengan LSTM lebih fair (apple-to-apple)')
    print('  karena keduanya menggunakan cakupan data historis yang sama.')
    print()
    print('REKOMENDASI UNTUK SKRIPSI:')
    print('  Ubah N_WINDOW_ARIMA dari 24 menjadi None (hapus pembatasan).')
    print('  Update seluruh tabel eksperimen N_WINDOW di Subbab 4.6.2 dengan')
    print('  menambahkan kolom N_WINDOW=None sebagai skenario baru.')
    print('  Update MAE/RMSE ARIMA di Tabel 4.47 dengan hasil Konfigurasi B.')
else:
    print('  -> Kedua konfigurasi menghasilkan MAE yang identik.')
    print('     Pertahankan N_WINDOW=24 karena lebih efisien secara komputasi.')
print()

# Rekapitulasi angka
print('='*70)
print('REKAPITULASI ANGKA UNTUK TABEL DI SKRIPSI')
print('='*70)
print(f'  MAE ARIMA Konfigurasi A (N_WINDOW=24)   : Rp{mae_A:,.0f}')
print(f'  RMSE ARIMA Konfigurasi A                : Rp{rmse_A:,.0f}')
print(f'  Precision@10 Konfigurasi A              : {prec_A:.3f}')
print(f'  Waktu Training Konfigurasi A            : {t_A:.1f} detik')
print()
print(f'  MAE ARIMA Konfigurasi B (N_WINDOW=None) : Rp{mae_B:,.0f}')
print(f'  RMSE ARIMA Konfigurasi B                : Rp{rmse_B:,.0f}')
print(f'  Precision@10 Konfigurasi B              : {prec_B:.3f}')
print(f'  Waktu Training Konfigurasi B            : {t_B:.1f} detik')

KESIMPULAN DAN REKOMENDASI

Kemenangan per bulan (berdasarkan error absolut terkecil):
  B (full-data): 7 bulan
  A (window-24): 5 bulan

Perbandingan MAE:
  Konfigurasi A (N_WINDOW=24)   : Rp2,467,639
  Konfigurasi B (N_WINDOW=None) : Rp2,462,224

  -> Konfigurasi B (full data) LEBIH BAIK (0.2% MAE lebih rendah).

INTERPRETASI:
  Menggunakan full data memberikan prediksi lebih akurat untuk ARIMA.
  Ini sekaligus membuat perbandingan dengan LSTM lebih fair (apple-to-apple)
  karena keduanya menggunakan cakupan data historis yang sama.

REKOMENDASI UNTUK SKRIPSI:
  Ubah N_WINDOW_ARIMA dari 24 menjadi None (hapus pembatasan).
  Update seluruh tabel eksperimen N_WINDOW di Subbab 4.6.2 dengan
  menambahkan kolom N_WINDOW=None sebagai skenario baru.
  Update MAE/RMSE ARIMA di Tabel 4.47 dengan hasil Konfigurasi B.

REKAPITULASI ANGKA UNTUK TABEL DI SKRIPSI
  MAE ARIMA Konfigurasi A (N_WINDOW=24)   : Rp2,467,639
  RMSE ARIMA Konfigurasi A                : Rp2,950,435
  Precision@10 Konfigura